In [1]:
from agents import Order, Restaurant, User, Driver
from environment import Environment
from simulation import Simulation, generate_orders, get_order_rate
from policies.dispatch import GreedyPolicy, HungarianPolicy
from policies.repositioning import StaticPolicy
from routing import distrito_tec, get_closest_place_node_id
import numpy as np
import random
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import matplotlib.cm as cm
import numpy as np
import osmnx as ox
import pandas as pd

sub_graph, routable_restaurants, residential_zones = distrito_tec()


In [2]:
import random
import numpy as np
random.seed(42)
np.random.seed(42)

In [3]:
routable_restaurants.head(5)

,element,id,geometry,amenity,brand,brand:wikidata,brand:wikipedia,cuisine,name,official_name,...,air_conditioning,website:menu,contact:instagram,phone,drive_through,building,fast_food,building:levels,height,nn
0,node,890657171,POINT (-100.29383 25.65435),cafe,Starbucks,Q37158,en:Starbucks,coffee_shop,Starbucks,Starbucks Coffee,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.433046e+09
1,node,890657529,POINT (-100.29377 25.65441),restaurant,NaN,NaN,NaN,NaN,Costeñito,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.433046e+09
2,node,890657855,POINT (-100.29355 25.65472),restaurant,NaN,NaN,NaN,NaN,Wings Army,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.433046e+09
3,node,890666591,POINT (-100.28436 25.64771),restaurant,NaN,NaN,NaN,NaN,Manhattan,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.049531e+09
4,node,890667172,POINT (-100.28432 25.64783),bar,NaN,NaN,NaN,NaN,La Rambla,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.049531e+09


In [4]:
routable_restaurants.shape

(50, 40)

In [5]:
residential_zones.head(5)

,element,id,geometry,landuse,name,place,addr:city,addr:street,building:levels,residential,operator,type
0,relation,9463437,"POLYGON ((-100.29348 25.66187, -100.2936 25.66...",residential,La Florida,NaN,NaN,NaN,NaN,NaN,NaN,multipolygon
1,way,163034600,"POLYGON ((-100.27483 25.62377, -100.27426 25.6...",residential,Contry,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,way,163034602,"POLYGON ((-100.27739 25.64568, -100.27511 25.6...",residential,Contry Tesoro,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,way,163034603,"POLYGON ((-100.28282 25.64297, -100.28238 25.6...",residential,Contry Lux,neighbourhood,NaN,NaN,NaN,NaN,NaN,NaN
4,way,163034605,"POLYGON ((-100.28052 25.64492, -100.27925 25.6...",residential,Las Musas,neighbourhood,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
routable_restaurants.iloc[[0]]

,element,id,geometry,amenity,brand,brand:wikidata,brand:wikipedia,cuisine,name,official_name,...,air_conditioning,website:menu,contact:instagram,phone,drive_through,building,fast_food,building:levels,height,nn
0,node,890657171,POINT (-100.29383 25.65435),cafe,Starbucks,Q37158,en:Starbucks,coffee_shop,Starbucks,Starbucks Coffee,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.433046e+09


| Scenario | Target simultaneous | Pool size |
| :--- | :--- | :--- |
| Undersupplied | 50 | ~72 |
| Balanced | 150 | ~215 |
| Oversupplied | 300 | ~430 |

In [7]:
#ORDER_RATE = 8    # orders/min off-peak  (~480/hr)
#ORDER_RATE = 25   # orders/min lunch peak (~1500/hr)  
#ORDER_RATE = 20   # orders/min dinner peak (~1200/hr)
#N_DRIVERS = 50    # undersupplied scenario
#N_DRIVERS = 100   # balanced
#N_DRIVERS = 200   # oversupplied
N_DRIVERS = 215

In [8]:
# =========================================================
# SCENARIO SETUP
# To swap strategies, change DISPATCH_POLICY / REPOSITION_POLICY.
# Nothing else in this file needs to change.
# =========================================================

# ---- Parameters ----
N_USERS           = 10000
#N_DRIVERS         = 50
#ORDER_RATE        = 15    # orders per minute, unused if dynamic wrapper in place
DISPATCH_INTERVAL = 3    # seconds between batch dispatches
START_HOUR = 11
WARMUP = 6
# ---- Swap policies here ----
DISPATCH_POLICY   = GreedyPolicy()  
REPOSITION_POLICY = StaticPolicy()                       # or RLPolicy() when ready

# ---------------------------------------------------------
# 1. Build environment + simulation
# ---------------------------------------------------------

env = Environment(sub_graph)

sim = Simulation(
    env=env,
    dispatch_policy=DISPATCH_POLICY,
    repositioning_policy=REPOSITION_POLICY,
    step_size=10,
    dispatch_interval=DISPATCH_INTERVAL,
    start_hour=START_HOUR-WARMUP # Used for dynamic demand wrapper
)

# ---------------------------------------------------------
# 2. Add restaurants
#    Ratings sampled uniformly [3.0, 5.0] so the MNL
#    restaurant-choice model has something to work with.
# ---------------------------------------------------------

for i in range(len(routable_restaurants)):

    res_node = get_closest_place_node_id(
        routable_restaurants.iloc[[i]],
        sub_graph
    )

    sim.add_restaurant(Restaurant(
        restaurant_id=i,
        location=res_node,
        rating=round(random.uniform(3.0, 5.0), 1),
        capacity=10,
        avg_prep_time=780,
        service_radius=5000,
    ))

print(f"{len(sim.restaurants)} restaurants loaded")

# ---------------------------------------------------------
# 3. Sample residential nodes for users / drivers
# ---------------------------------------------------------

sampled_zones = residential_zones.sample(N_USERS, replace=True)

residential_nodes = [
    get_closest_place_node_id(sampled_zones.iloc[[i]], sub_graph)
    for i in range(N_USERS)
]

# ---------------------------------------------------------
# 4. Create users
# ---------------------------------------------------------

for i in range(N_USERS):
    sim.add_user(User(user_id=i, location=residential_nodes[i]))

print(f"{len(sim.users)} users created")

# ---------------------------------------------------------
# 5. Create drivers
# ---------------------------------------------------------

for i in range(N_DRIVERS):
    sim.add_driver(Driver(
        driver_id=i,
        location_node=random.choice(residential_nodes),
        speed=None,
    ))

print(f"{len(sim.drivers)} drivers created")
print(f"Dispatch  : {sim.dispatch_policy.__class__.__name__}")
print(f"Reposition: {sim.repositioning_policy.__class__.__name__}")
print("Simulation ready.")


50 restaurants loaded
10000 users created
215 drivers created
Dispatch  : GreedyPolicy
Reposition: StaticPolicy
Simulation ready.


## Dynamic driver fleet 

In [9]:
def schedule_driver_shifts(sim: Simulation, residential_nodes: list[int]):
    """Schedules driver shift changes throughout the simulated day.

    Models two shift cohorts with staggered start/end times to avoid
    simultaneous mass availability changes. Reflects gig platform
    dynamics where drivers self-select into peak-hour windows.

    Shift structure (simulated wall clock):
        Morning shift : 10:00 - 15:00  (~30% of fleet)
        Evening shift : 17:00 - 23:00  (~50% of fleet)
        Always-on     :                (~20% of fleet)

    Args:
        sim: Running Simulation instance.
        residential_nodes: Pool of valid spawn nodes for incoming drivers.
    """


    drivers        = list(sim.drivers.values())
    n              = len(drivers)
    morning_cohort = drivers[:int(n * 0.30)]
    evening_cohort = drivers[int(n * 0.30):int(n * 0.80)]

    for morn_driver in morning_cohort:
        jitter  = random.uniform(-900, 900)
        start_s = max(0.0, (09.00 - sim.start_hour) * 3600 + jitter)
        end_s   = max(0.0, (15.0 - sim.start_hour) * 3600 + jitter)
        if end_s > 0:
            if start_s >= 0:
                sim.schedule_event(start_s, 'enable_driver', morn_driver.id)
            sim.schedule_event(end_s, 'disable_driver', morn_driver.id)

    for eve_driver in evening_cohort:
        jitter  = random.uniform(-900, 900)
        start_s = max(0.0, (17.0 - sim.start_hour) * 3600 + jitter)
        end_s   = max(0.0, (23.0 - sim.start_hour) * 3600 + jitter)
        if end_s >= 0:
            if start_s > 0:
                sim.schedule_event(start_s, 'enable_driver', eve_driver.id)
            sim.schedule_event(end_s, 'disable_driver', eve_driver.id)

In [10]:
# 1. Disable all except always-on cohort
for i, d in enumerate(sim.drivers.values()):
    if i < int(N_DRIVERS * 0.80):
        d.available = False
        sim.idle_drivers.discard(d.id)
# 2. Schedule shift changes
schedule_driver_shifts(sim, residential_nodes)

In [11]:
SHOW_TRAILS = False

## Simualtion warmup

In [12]:
print("Morning cohort before warmup:")
for d in list(sim.drivers.values())[:3]:
    print(f"  driver={d.id} available={d.available}")

print("\nFirst 5 scheduled events:")
for t, e, p in sorted(sim._schedule)[:5]:
    print(f"  t={t:.0f} {e} driver={p}")

sim.run_until(3600 * WARMUP)
# Warmup — not recorded in metrics
sim.run_until(3600*WARMUP)

## Diagnostic
print(f"At end of warmup t={sim.current_time}")
print(f"Active: {sum(1 for d in sim.drivers.values() if d.available)}")
print(f"Scheduled events remaining: {len(sim._schedule)}")
for t, e, p in sorted(sim._schedule)[:10]:
    print(f"  t={t:.0f} {e} driver={p}")

# ADD THIS RIGHT HERE
print("\n=== Enable events remaining ===")
enable_events = [(t, e, p) for t, e, p in sim._schedule if 'enable' in e]
print(f"Count: {len(enable_events)}")
for t, e, p in sorted(enable_events)[:20]:
    print(f"  t={t:.0f} {e} driver={p}")

# Reset metrics but keep system state
for o in list(sim.orders.values()):
    sim.orders.pop(o.id)
sim.order_id_counter = 1
sim._pending_set.clear()
sim.pending_orders.clear()

Morning cohort before warmup:
  driver=0 available=False
  driver=1 available=False
  driver=2 available=False

First 5 scheduled events:
  t=13511 enable_driver driver=37
  t=13549 enable_driver driver=57
  t=13549 enable_driver driver=60
  t=13552 enable_driver driver=49
  t=13616 enable_driver driver=25
At end of warmup t=21600.0
Active: 107
Scheduled events remaining: 280
  t=35111 disable_driver driver=37
  t=35149 disable_driver driver=57
  t=35149 disable_driver driver=60
  t=35152 disable_driver driver=49
  t=35216 disable_driver driver=25
  t=35239 disable_driver driver=16
  t=35330 disable_driver driver=55
  t=35344 disable_driver driver=62
  t=35349 disable_driver driver=21
  t=35397 disable_driver driver=53

=== Enable events remaining ===
Count: 108
  t=42318 enable_driver driver=154
  t=42335 enable_driver driver=74
  t=42337 enable_driver driver=84
  t=42346 enable_driver driver=99
  t=42352 enable_driver driver=145
  t=42391 enable_driver driver=64
  t=42437 enable_driv

In [13]:
print("Morning cohort state after reset:")
morning_ids = [d.id for d in list(sim.drivers.values())[:int(N_DRIVERS * 0.30)]]
for did in morning_ids[:5]:
    d = sim.drivers[did]
    print(f"  driver={did} available={d.available} status={d.status}")

Morning cohort state after reset:
  driver=0 available=True status=IDLE
  driver=1 available=True status=IDLE
  driver=2 available=True status=IDLE
  driver=3 available=True status=IDLE
  driver=4 available=True status=IDLE


In [14]:
# After warmup reset, before animation
print(f"Immediately after reset:")
print(f"Active: {sum(1 for d in sim.drivers.values() if d.available)}")

Immediately after reset:
Active: 107


In [15]:
%matplotlib inline


# --- Map canvas ---
fig, ax = ox.plot_graph(
    sub_graph, bgcolor='k', edge_color='#333333',
    node_size=0, edge_linewidth=0.5, show=False, close=False
)

# --- Static restaurant markers ---
restaurant_lons = [sub_graph.nodes[r.location]['x'] for r in sim.restaurants.values()]
restaurant_lats = [sub_graph.nodes[r.location]['y'] for r in sim.restaurants.values()]

ax.scatter(
    restaurant_lons, restaurant_lats,
    s=120, c='red', marker='^',
    edgecolors='white', linewidth=1.2,
    zorder=12, label='Restaurants'
)

# --- Driver visual elements ---
drivers_to_watch = list(sim.drivers.values())
if not drivers_to_watch:
    raise ValueError("No drivers found — check setup cell.")

colors = cm.rainbow(np.linspace(0, 1, len(drivers_to_watch)))
dots   = ax.scatter([], [], c=[], s=60, zorder=10, edgecolors='white', linewidth=0.8)

if SHOW_TRAILS:
    lines = [ax.plot([], [], color=colors[i], lw=2.5, alpha=0.7, zorder=9)[0]
             for i in range(len(drivers_to_watch))]
else:
    lines = []

trail_history = [[] for _ in drivers_to_watch] if SHOW_TRAILS else None

time_text = ax.text(
    0.02, 0.95, '', transform=ax.transAxes,
    color='white', fontsize=11, fontweight='bold',
    verticalalignment='top', fontfamily='monospace'
)

# --- Animation update ---
def update(frame):
    # Dynamic order rate, based on get_order_rate and wall clock hour 
    order_rate = get_order_rate(sim)
    # if you need a harcoded value use 
    #order_rate = ORDER_RATE
    generate_orders(sim, rate_per_minute=order_rate)
    sim.run_tick()

    current_lons, current_lats = [], []
    for i, driver in enumerate(drivers_to_watch):
        if driver.coords != (0.0, 0.0):
            lon, lat = driver.coords[1], driver.coords[0]
        else:
            node_data = sub_graph.nodes[driver.location]
            lon, lat  = node_data['x'], node_data['y']

        current_lons.append(lon)
        current_lats.append(lat)

        if SHOW_TRAILS:
            trail_history[i].append((lon, lat))
            h_x, h_y = zip(*trail_history[i])
            lines[i].set_data(h_x, h_y)

    dots.set_offsets(np.c_[current_lons, current_lats])

    driver_colors = []
    for driver in drivers_to_watch:
        if not driver.available:
            driver_colors.append('gray')
        elif driver.status == 'IDLE':
            driver_colors.append('white')
        else:
            driver_colors.append(colors[drivers_to_watch.index(driver)])

    dots.set_color(driver_colors)

    m = sim.metrics_snapshot()
    s = m['orders_by_status']
    hud = (
        f"t={int(sim.current_time)}s  [{m['dispatch_policy']} / {m['repositioning_policy']}]\n"
        f"Simulation time {sim.wall_clock_display}\n"
        f"PREP:{s['PREPARING']}  READY:{s['READY']}  "
        f"PICKUP:{s['PICKED_UP']}  DONE:{s['DELIVERED']}\n"
        f"Idle drivers:{m['idle_drivers']}\n"
        f"Deactivated drivers:{m['deactivated_drivers']}\n"
        f"Active drivers:{m['active_drivers']}\n"
        f"avg e2e:{int(m['avg_end_to_end_s'] or 0)}s"
    )
    time_text.set_text(hud)

    return ([dots, time_text] + lines) if SHOW_TRAILS else [dots, time_text]

# frames=X * step_size=Y => XY simulated seconds
ani = FuncAnimation(fig, update, frames=8640, interval=50, blit=True, repeat=False)
plt.close()
ani.save("sim_1dgreedy.mp4",writer='ffmpeg', fps=30)


In [16]:
# =========================================================
# POST-RUN METRICS
# Run after animation finishes or after sim.run_until(T).
# =========================================================


m = sim.metrics_snapshot()
print(f"Dispatch policy    : {m['dispatch_policy']}")
print(f"Reposition policy  : {m['repositioning_policy']}")
print(f"Simulated time     : {m['time']:.0f}s")
print(f"Total orders       : {m['total_orders']}")
print(f"Delivered          : {m['n_delivered']}")
print(f"Pending unassigned : {m['pending_unassigned']}")
print(f"Idle drivers       : {m['idle_drivers']}")
print(f"Deactivated drivers: {m['deactivated_drivers']}")
print(f"Avg end-to-end     : {m['avg_end_to_end_s']:.1f}s"     if m['avg_end_to_end_s']     else "Avg end-to-end     : n/a")
print(f"Avg food wait      : {m['avg_food_wait_s']:.1f}s"      if m['avg_food_wait_s']      else "Avg food wait      : n/a")
print(f"Avg dispatch delay : {m['avg_dispatch_delay_s']:.1f}s"  if m['avg_dispatch_delay_s']  else "Avg dispatch delay : n/a")

# Per-order ledger — trace anything suspicious here
records = [
    {
        'order_id'         : o.id,
        'user_id'          : o.user_id,
        'driver_id'        : o.driver_id,
        'restaurant_id'    : o.restaurant_id,
        'start_time'       : o.start_time,
        'assigned_time'    : o.assigned_time,
        'pickup_time'      : o.pickup_time,
        'delivered_time'   : o.delivered_time,
        'end_to_end_s'     : o.end_to_end_time,
        'food_wait_s'      : o.food_wait_time,
        'dispatch_delay_s' : o.dispatch_delay,
        'status'           : o.status,
        'prep_time': o.prep_time,
    }
    for o in sim.orders.values()
]

df = pd.DataFrame(records)



Dispatch policy    : GreedyPolicy
Reposition policy  : StaticPolicy
Simulated time     : 108030s
Total orders       : 3531
Delivered          : 3472
Pending unassigned : 15
Idle drivers       : 0
Deactivated drivers: 172
Avg end-to-end     : 865.3s
Avg food wait      : 19.7s
Avg dispatch delay : 13.4s


In [17]:
df.to_csv("resultsgreedy.csv",index=0)

In [18]:
print(sim.current_time, sim.wall_clock_hour)
idle = len(sim.idle_drivers)
disabled = sum(1 for d in sim.drivers.values() if not d.available)
busy = sum(1 for d in sim.drivers.values() if d.status != 'IDLE')
print(f"idle={idle} disabled={disabled} busy={busy}")
ready_orders = [o for o in sim.orders.values() if o.status == 'READY']
assigned = sum(1 for o in ready_orders if o.driver_id is not None)
unassigned = sum(1 for o in ready_orders if o.driver_id is None)
print(f"ready assigned={assigned} unassigned={unassigned}")
ready_unassigned_ids = {o.id for o in ready_orders if o.driver_id is None}
in_pending = ready_unassigned_ids & sim._pending_set
print(f"unassigned READY in pending_set={len(in_pending)}")

108030.0 11.008333333333333
idle=0 disabled=172 busy=44
ready assigned=3 unassigned=12
unassigned READY in pending_set=12


## Buckets by hour

In [19]:

delivered = [o for o in sim.orders.values() if o.status == 'DELIVERED']

import numpy as np
import pandas as pd

df_delivered = pd.DataFrame([{
    'assigned_time': o.assigned_time,
    'dispatch_delay': o.dispatch_delay,
    'end_to_end': o.end_to_end_time,
    'start_time': o.start_time,
} for o in delivered])

# bucket by simulated hour
df_delivered['hour'] = (df_delivered['start_time'] / 3600 + sim.start_hour) % 24 
df_delivered["day_bucket"] = (df_delivered['start_time'] / 3600 + sim.start_hour) // 24 
df_delivered['hour_bucket'] = df_delivered['hour'].astype(int)

print(df_delivered.groupby(['day_bucket','hour_bucket'])[['dispatch_delay','end_to_end']].mean().round(0))

                        dispatch_delay  end_to_end
day_bucket hour_bucket                            
0.0        11                     10.0       858.0
           12                     10.0       875.0
           13                     10.0       849.0
           14                     10.0       866.0
           15                     10.0       859.0
           16                     10.0       878.0
           17                     10.0       859.0
           18                     10.0      1021.0
           19                     10.0       875.0
           20                     10.0       865.0
           21                     10.0       862.0
           22                     10.0       708.0
           23                     10.0       741.0
1.0        0                      10.0       728.0
           1                      10.0       794.0
           2                      10.0       759.0
           3                      10.0       737.0
           4                   

In [20]:
df

,order_id,user_id,driver_id,restaurant_id,start_time,assigned_time,pickup_time,delivered_time,end_to_end_s,food_wait_s,dispatch_delay_s,status,prep_time
0,1,9017,51.0,17,21610.0,21620.0,22160.0,22600.0,990.0,0.000000,10.0,DELIVERED,873.887756
1,2,4118,41.0,38,21610.0,21620.0,21930.0,22530.0,920.0,0.000000,10.0,DELIVERED,908.332239
2,3,3033,214.0,13,21620.0,21630.0,22120.0,22850.0,1230.0,0.000000,10.0,DELIVERED,913.403387
3,4,9659,213.0,45,21620.0,21630.0,22040.0,22450.0,830.0,0.000000,10.0,DELIVERED,708.937613
4,5,3727,212.0,0,21650.0,21660.0,22170.0,22910.0,1260.0,134.936296,10.0,DELIVERED,385.063704
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3526,3527,8191,197.0,14,107940.0,108020.0,NaN,NaN,NaN,NaN,80.0,PREPARING,395.549861
3527,3528,6744,199.0,17,107960.0,108030.0,NaN,NaN,NaN,NaN,70.0,PREPARING,685.738779
3528,3529,6522,NaN,13,107980.0,NaN,NaN,NaN,NaN,NaN,NaN,PREPARING,1133.606183
3529,3530,9706,NaN,39,107980.0,NaN,NaN,NaN,NaN,NaN,NaN,PREPARING,909.577156


In [21]:
# Driver availability timeline
import pandas as pd

timeline = []
for t, event, payload in sim._schedule:
    timeline.append({'trigger': t, 'event': event, 'driver': payload})

# Plus any already-fired events won't be in _schedule
# So also check current state
print(f"Total drivers: {len(sim.drivers)}")
print(f"Available: {sum(1 for d in sim.drivers.values() if d.available)}")
print(f"Unavailable: {sum(1 for d in sim.drivers.values() if not d.available)}")
print(f"Remaining scheduled events: {len(sim._schedule)}")
print(pd.DataFrame(timeline).groupby('event')['trigger'].describe().round(0) if timeline else "schedule empty")

Total drivers: 215
Available: 43
Unavailable: 172
Remaining scheduled events: 0
schedule empty


In [22]:
# Order volume by hour (not just delivered — all orders)
all_orders = pd.DataFrame([{
    'start_time': o.start_time,
    'status': o.status,
} for o in sim.orders.values()])
all_orders['hour'] = (all_orders['start_time'] / 3600 + sim.start_hour) % 24
all_orders["day_bucket"] = (all_orders['start_time'] / 3600 + sim.start_hour) // 24
all_orders['hour_bucket'] = all_orders['hour'].astype(int)
print(all_orders.groupby(['hour_bucket','day_bucket'])['status'].value_counts().unstack(fill_value=0))

status                  DELIVERED  PICKED_UP  PREPARING  READY
hour_bucket day_bucket                                        
0           1.0                28          0          0      0
1           1.0                22          0          0      0
2           1.0                27          0          0      0
3           1.0                35          0          0      0
4           1.0                29          0          0      0
5           1.0                24          0          0      0
6           1.0                91          0          0      0
7           1.0                96          0          0      0
8           1.0                89          0          0      0
9           1.0                92          1          0      0
10          1.0               136         22         20     15
11          0.0               262          0          0      0
12          0.0               307          0          0      0
13          0.0               297          0          0

In [23]:
# Diagnose always-on cohort
print("Always-on driver IDs:")
always_on = [d.id for d in sim.drivers.values() if d.available]
print(sorted(always_on))
print(f"\nAlways-on count: {len(always_on)}")
print(f"Idle at end of sim: {sum(1 for d in sim.drivers.values() if d.available and d.status == 'IDLE')}")

Always-on driver IDs:
[172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214]

Always-on count: 43
Idle at end of sim: 0


In [24]:
# How many drivers are active and idle at each problem hour?
import pandas as pd

# You'll need to instrument this during the run
# For now, check end state
print(f"Active at end: {sum(1 for d in sim.drivers.values() if d.available)}")
print(f"Busy at end: {sum(1 for d in sim.drivers.values() if d.status != 'IDLE' and d.available)}")

Active at end: 43
Busy at end: 43


In [25]:
# Run after greedy simulation
print(f"Total orders placed: {len(sim.orders)}")
print(f"Delivered: {len([o for o in sim.orders.values() if o.status == 'DELIVERED'])}")
print(f"Stuck READY (no driver): {len([o for o in sim.orders.values() if o.status == 'READY' and o.driver_id is None])}")
print(f"Stuck READY (has driver): {len([o for o in sim.orders.values() if o.status == 'READY' and o.driver_id is not None])}")

Total orders placed: 3531
Delivered: 3472
Stuck READY (no driver): 12
Stuck READY (has driver): 3


In [26]:
print(f"Total orders placed: {len(sim.orders)}")
print(f"Sim end time: {sim.current_time}")
print(f"Warmup seconds: {3600 * WARMUP}")
print(f"Recording frames: 8640")
print(f"Step size: {sim.step_size}")
print(f"Expected sim time: {3600 * WARMUP + 8640 * sim.step_size}")

Total orders placed: 3531
Sim end time: 108030.0
Warmup seconds: 21600
Recording frames: 8640
Step size: 10
Expected sim time: 108000
